<a href="https://colab.research.google.com/github/asharali2503/Exoplanet-discovery-analysis/blob/main/exoplanet_hunt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install lightkurve -q

In [ ]:
import lightkurve as lk
import numpy as np


In [ ]:
search = lk.search_lightcurve("Pi Mensae", author="SPOC", exptime=120)
search

In [ ]:
lc = search[0:3].download_all().stitch().remove_nans()
lc.plot()

In [ ]:
flat = lc.flatten(window_length=401)
flat = flat.remove_outliers(sigma_upper=4, sigma_lower=np.inf)
flat.plot()


In [ ]:
period_grid = np.linspace(0.5, 15, 20000)
bls = flat.to_periodogram(method="bls", period=period_grid, frequency_factor=500)
bls.plot()

In [ ]:
planet_period = bls.period_at_max_power
planet_t0 = bls.transit_time_at_max_power
planet_depth = bls.depth_at_max_power
print(f"Best period: {planet_period:.4f}")
print(f"Depth: {planet_depth:.5f} ({planet_depth * 100:.3f} percent)")
print(f"Power score: {bls.max_power:.4f}")

In [ ]:
folded = flat.fold(period=planet_period, epoch_time=planet_t0)
ax = folded.scatter(s=1)
folded.bin(time_bin_size=10 / 60 / 24).plot(ax=ax, color="red", lw=2)

In [ ]:
hunt("HD 63433")
hunt("AU Mic")


In [ ]:
half = flat.fold(period=planet_period, epoch_time=planet_t0 + planet_period / 2)
half.scatter(s=1)

In [ ]:
ax = folded[folded.odd_mask].scatter(s=1, label="odd transits")
folded[folded.even_mask].scatter(ax=ax, s=1, color="red", label="even transits")

In [ ]:
star_radius_suns = 1.1 # put your star's radius here, in Suns
planet_radius_suns = star_radius_suns * np.sqrt(planet_depth)
print(f"Planet radius: {planet_radius_suns * 109.2:.2f} Earths")
print(f"Planet radius: {planet_radius_suns * 9.95:.2f} Jupiters")

### 🚀 The Hunt Begins

In [ ]:
hunt("WASP-18")

In [ ]:
hunt("LHS 1140")

In [ ]:
# A list of random star targets to scan automatically
target_list = [
    "HD 106906",
    "HD 209458",
    "HD 219134",
    "TIC 281541555",
    "TIC 27081059"
]

# The automation loop
for star in target_list:
    print(f"\n====================================")
    print(f"🚀 NOW SCANNING: {star}")
    print(f"====================================")

    # We use 'try' so if one star fails or has no data, the loop doesn't crash
    try:
        hunt(star)
    except Exception as e:
        print(f"Could not scan {star}. Error: {e}")

In [ ]:
# Install astroquery in Colab if it isn't already there
!pip install astroquery

from astroquery.mast import Observations

print("📡 Connecting to NASA MAST servers...")

# We use query_criteria to efficiently pull TESS data directly from the archive
# Here we are pulling time-series data specifically from TESS Sector 26
obs_table = Observations.query_criteria(
    provenance_name="QLP",
    sequence_number=26
)

# Extract the TIC IDs from the target_name column
# We use a 'set' to remove duplicates, then slice the first 1000 targets
raw_tics = list(set(obs_table['target_name']))[:1000]

# Format them so your hunt() function can read them perfectly (e.g., "TIC 12345678")
target_list = [f"TIC {tic}" for tic in raw_tics if str(tic).isdigit()]

print(f"✅ Successfully downloaded {len(target_list)} fresh TIC IDs to scan!")
print("Here is a sneak peek at the first 5 targets:")
print(target_list[:5])

In [ ]:
import pandas as pd

print("📡 Downloading live TESS targets from NASA ExoFOP...")

# Fetch live public catalog of TESS planet candidates
url = "https://exofop.ipac.caltech.edu/tess/download_toi.php?sort=toi&output=csv"
df = pd.read_csv(url)

# Grab 1,000 unconfirmed candidate TIC IDs
candidates = df[df['TFOPWG Disposition'] == 'PC']
raw_tics = candidates['TIC ID'].dropna().unique()[:1000]

target_list = [f"TIC {int(tic)}" for tic in raw_tics]

print(f"✅ Downloaded {len(target_list)} fresh TIC IDs to scan!")
print("First 5 targets in queue:", target_list[:5])

In [ ]:
# Loop through the first 5 newly downloaded targets and run your pipeline
for star in target_list[:5]:
    print(f"\n====================================")
    print(f"🚀 AUTOMATED SCAN: {star}")
    print(f"====================================")
    try:
        hunt(star)
    except Exception as e:
        print(f"Skipping {star} due to missing archive data: {e}")

In [ ]:
def hunt(star, min_period=0.5, max_period=15, plot=True):
    # Fetch the data
    search = lk.search_lightcurve(star, author="SPOC", exptime=120)
    if len(search) == 0:
        search = lk.search_lightcurve(star, author="QLP")
    if len(search) == 0:
        if plot: print(f"No TESS data for {star}, try another star")
        return None, None, None

    # Clean and process
    lc = search[0:3].download_all().stitch().remove_nans()
    flat = lc.flatten(window_length=401)
    flat = flat.remove_outliers(sigma_upper=4, sigma_lower=np.inf)

    # Run the Box Least Squares (BLS) math
    grid = np.linspace(min_period, max_period, 20000)
    bls = flat.to_periodogram(method="bls", period=grid, frequency_factor=500)
    period = bls.period_at_max_power
    t0 = bls.transit_time_at_max_power
    depth = bls.depth_at_max_power
    power = bls.max_power

    # Only plot if the loop tells it to
    if plot:
        print(f"Star: {star} | TIC: {lc.meta.get('TICID', 'see search table')}")
        print(f"Best period: {period:.4f}")
        print(f"Depth: {depth:.5f} ({depth * 100:.3f} percent)")
        print(f"Power score: {power:.4f}")
        folded = flat.fold(period=period, epoch_time=t0)
        ax = folded.scatter(s=1)
        folded.bin(time_bin_size=10 / 60 / 24).plot(ax=ax, color="red", lw=2)

    # Send the math back to the loop for filtering
    return depth.value, power.value, period.value

In [ ]:
import matplotlib.pyplot as plt

print("==================================================")
print("🚀 INITIATING MASSIVE AUTOMATED SCAN...")
print("Scanning targets in queue. This will filter and plot only strong candidates.")
print("==================================================\n")

# Loop through every star in your downloaded target list
for star in target_list:
    try:
        # Run the math silently (plot=False)
        depth, power, period = hunt(star, plot=False)

        # If the star has no data, skip it
        if depth is None:
            continue

        # THE FILTER: High power score (strong signal) and planet-like depth (< 3%)
        if power > 500 and depth < 0.03:
            print(f"🚨 PRIME CANDIDATE FOUND: {star}")
            print(f"Depth: {depth:.4f} | Power: {power:.2f} | Period: {period:.2f} days")

            # Run again WITH the plot turned on to view the graph
            hunt(star, plot=True)
            plt.show()  # <--- THIS FORCES THE GRAPH TO APPEAR INSTANTLY!
            print("---------------------------------------------------\n")

    except Exception as e:
        # If a star fails or encounters an error, silently skip it
        pass

In [ ]:
import os
import shutil
import matplotlib.pyplot as plt

# 1. Put the TIC IDs you want to save right here (I added the 4 from your screenshots!)
target_list = ["130415266", "282375913", "179580045", "92226327"] # Add any others from your text output!

# 2. Create the folder to hold the images
os.makedirs("exoplanet_discoveries", exist_ok=True)

print("🚀 INITIATING QUICK EXTRACTION...")

# 3. Run the scanner just for these specific targets
for star in target_list:
    try:
        # We run the hunt function and plot it
        depth, power, period = hunt(star, plot=True)

        # Save the graph as an image file BEFORE showing it
        plt.savefig(f"exoplanet_discoveries/TIC_{star}.png", bbox_inches='tight')
        plt.show()
        print(f"✅ Saved graph for TIC {star}\n")

    except Exception as e:
        pass

# 4. Automatically zip the folder when it finishes!
shutil.make_archive("discoveries", 'zip', "exoplanet_discoveries")

print("📦 SUCCESS! discoveries.zip is ready to download!")